In [1]:
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path("../data/raw")

stations = [
    "toronto",
    "ottawa",
    "windsor",
    "sudbury",
    "thunder_bay"
]

weather_data = {}

for station in stations:
    file_path = DATA_DIR / f"weather_{station}_2025.json"

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    df = pd.DataFrame(
        feature["properties"]
        for feature in data["features"]
    )

    df["timestamp"] = pd.to_datetime(df["LOCAL_DATE"])
    df["temperature"] = pd.to_numeric(
        df["TEMP"],
        errors="coerce"
    )

    df = df[
        ["timestamp", "temperature"]
    ].sort_values("timestamp")

    weather_data[station] = df

In [2]:
expected_hours = pd.date_range(
    "2025-01-01 00:00:00",
    "2025-12-31 23:00:00",
    freq="h"
)

In [3]:
cleaned_weather = {}

for station, df in weather_data.items():
    full = (
        df.set_index("timestamp")
        .reindex(expected_hours)
    )

    full["temperature"] = full["temperature"].interpolate(
        method="time",
        limit=6,
        limit_area="inside"
    )

    full.index.name = "timestamp"
    full = full.reset_index()

    cleaned_weather[station] = full

In [4]:
for station, df in cleaned_weather.items():
    print(
        station,
        "missing temperatures:",
        df["temperature"].isna().sum()
    )

toronto missing temperatures: 0
ottawa missing temperatures: 0
windsor missing temperatures: 0
sudbury missing temperatures: 0
thunder_bay missing temperatures: 0


In [5]:
frames = []

for station, df in cleaned_weather.items():
    station_df = df.copy()
    station_df["station"] = station

    frames.append(
        station_df[
            ["timestamp", "station", "temperature"]
        ]
    )

weather_all = pd.concat(
    frames,
    ignore_index=True
)

In [6]:
print("Shape:", weather_all.shape)

print(
    "Missing temperatures:",
    weather_all["temperature"].isna().sum()
)

print(
    "Duplicate keys:",
    weather_all.duplicated(
        subset=["timestamp", "station"]
    ).sum()
)

Shape: (43800, 3)
Missing temperatures: 0
Duplicate keys: 0
